In [1]:
import copy
import logging
import sys
import yaml

import numpy as np

import torch
import torch.multiprocessing as mp
import torch.nn.functional as F
from torch.nn.parallel import DistributedDataParallel

from ijepa.src.masks.random import MaskCollator
from ijepa.src.masks.multiblock import MaskCollator as MBMaskCollator
from ijepa.src.masks.utils import apply_masks
from ijepa.src.utils.distributed import (
    init_distributed,
    AllReduce
)
from ijepa.src.utils.logging import (
    CSVLogger,
    gpu_timer,
    grad_logger,
    AverageMeter)
from ijepa.src.utils.tensors import repeat_interleave_batch
from ijepa.src.datasets.imagenet1k import make_imagenet1k

from ijepa.src.helper import (
    load_checkpoint,
    init_model,
    init_opt)
from ijepa.src.transforms import make_transforms
from model import init_model, VisionTransformer

device = 7

encoder1 = VisionTransformer(in_chans=2).to(device)
encoder2 = VisionTransformer(in_chans=12).to(device)
target_encoder1 = VisionTransformer(in_chans=12).to(device)
target_encoder2 = VisionTransformer(in_chans=2).to(device)

for p in target_encoder1.parameters():
    p.requires_grad = False
for p in target_encoder2.parameters():
    p.requires_grad = False

In [5]:
globals().get('encoder1')

VisionTransformer(
  (patch_embed): PatchEmbed(
    (proj): Conv2d(2, 768, kernel_size=(16, 16), stride=(16, 16))
  )
  (blocks): ModuleList(
    (0-11): 12 x Block(
      (norm1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (attn): Attention(
        (qkv): Linear(in_features=768, out_features=2304, bias=True)
        (attn_drop): Dropout(p=0.0, inplace=False)
        (proj): Linear(in_features=768, out_features=768, bias=True)
        (proj_drop): Dropout(p=0.0, inplace=False)
      )
      (drop_path): Identity()
      (norm2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (mlp): MLP(
        (fc1): Linear(in_features=768, out_features=3072, bias=True)
        (act): GELU(approximate='none')
        (fc2): Linear(in_features=3072, out_features=768, bias=True)
        (drop): Dropout(p=0.0, inplace=False)
      )
    )
  )
  (norm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
)

In [29]:
collator = MaskCollator()

input = [torch.randn(2,224,224) for _ in range(8)]

batch, mask_enc, mask_pred = collator(input)
batch = batch.to(device)
mask_enc = mask_enc[0].to(device)
print(len(mask_pred))
mask_pred = mask_pred[0].to(device)
print(sorted(list(mask_pred[0].cpu().numpy())))
print(sorted(list(mask_enc[0].cpu().numpy())))
z1_target = encoder1(batch)
z1_target = F.layer_norm(z1_target, (z1_target.size(-1),))
z1_target = apply_masks(z1_target, mask_pred)

print(z1_target.shape)
z1_target = repeat_interleave_batch(z1_target, len(z1_target), repeat=len(mask_enc))
print(z1_target.shape)

1
[np.int64(0), np.int64(1), np.int64(2), np.int64(4), np.int64(5), np.int64(9), np.int64(10), np.int64(11), np.int64(12), np.int64(13), np.int64(14), np.int64(15), np.int64(17), np.int64(20), np.int64(25), np.int64(27), np.int64(28), np.int64(29), np.int64(30), np.int64(31), np.int64(34), np.int64(40), np.int64(42), np.int64(44), np.int64(47), np.int64(49), np.int64(54), np.int64(55), np.int64(56), np.int64(59), np.int64(61), np.int64(62), np.int64(69), np.int64(70), np.int64(71), np.int64(72), np.int64(73), np.int64(74), np.int64(76), np.int64(78), np.int64(79), np.int64(82), np.int64(86), np.int64(87), np.int64(88), np.int64(89), np.int64(90), np.int64(92), np.int64(94), np.int64(95), np.int64(97), np.int64(99), np.int64(100), np.int64(102), np.int64(105), np.int64(108), np.int64(111), np.int64(115), np.int64(117), np.int64(120), np.int64(122), np.int64(123), np.int64(125), np.int64(127), np.int64(128), np.int64(129), np.int64(131), np.int64(133), np.int64(135), np.int64(136), np.in

In [30]:
collator = MBMaskCollator()

input = [torch.randn(2,224,224) for _ in range(8)]

batch, mask_enc, mask_pred = collator(input)
batch = batch.to(device)
print(len(mask_pred))

2


In [31]:
mask_pred

[tensor([[  1,   2,   3,   4,   5,   6,   7,   8,  15,  16,  17,  18,  19,  20,
           21,  22,  29,  30,  31,  32,  33,  34,  35,  36,  43,  44,  45,  46,
           47,  48,  49,  50,  57,  58,  59,  60,  61,  62,  63,  64,  71,  72,
           73,  74,  75,  76,  77,  78,  85,  86,  87,  88,  89,  90,  91,  92,
           99, 100, 101, 102, 103, 104, 105, 106, 113, 114, 115, 116, 117, 118,
          119, 120, 127, 128, 129, 130, 131, 132, 133, 134, 141, 142, 143, 144,
          145, 146, 147, 148, 155, 156, 157, 158, 159, 160, 161, 162, 169, 170,
          171, 172, 173, 174, 175, 176],
         [  2,   3,   4,   5,   6,   7,   8,   9,  16,  17,  18,  19,  20,  21,
           22,  23,  30,  31,  32,  33,  34,  35,  36,  37,  44,  45,  46,  47,
           48,  49,  50,  51,  58,  59,  60,  61,  62,  63,  64,  65,  72,  73,
           74,  75,  76,  77,  78,  79,  86,  87,  88,  89,  90,  91,  92,  93,
          100, 101, 102, 103, 104, 105, 106, 107, 114, 115, 116, 117, 118, 119,

In [32]:
mask_enc

[tensor([[ 10,  11,  24,  25,  38,  39,  52,  53,  66,  67,  80],
         [ 28,  29,  42,  43,  56,  57,  70,  71,  84,  85,  98],
         [  0,   1,   2,  14,  15,  16,  28,  29,  30,  42,  43],
         [ 40,  54,  68,  82,  96, 110, 124, 138, 152, 166, 180],
         [  0,   1,  14,  15,  28,  29,  42,  43,  56,  57,  70],
         [ 22,  23,  24,  25,  36,  37,  38,  39,  50,  51,  52],
         [  8,   9,  10,  11,  12,  22,  23,  24,  25,  26,  36],
         [ 22,  23,  24,  25,  26,  36,  37,  38,  39,  40,  50]])]

In [35]:
mask_pred[0][0], mask_pred[1][0]

(tensor([  1,   2,   3,   4,   5,   6,   7,   8,  15,  16,  17,  18,  19,  20,
          21,  22,  29,  30,  31,  32,  33,  34,  35,  36,  43,  44,  45,  46,
          47,  48,  49,  50,  57,  58,  59,  60,  61,  62,  63,  64,  71,  72,
          73,  74,  75,  76,  77,  78,  85,  86,  87,  88,  89,  90,  91,  92,
          99, 100, 101, 102, 103, 104, 105, 106, 113, 114, 115, 116, 117, 118,
         119, 120, 127, 128, 129, 130, 131, 132, 133, 134, 141, 142, 143, 144,
         145, 146, 147, 148, 155, 156, 157, 158, 159, 160, 161, 162, 169, 170,
         171, 172, 173, 174, 175, 176]),
 tensor([  2,   3,   4,   5,   6,   7,   8,   9,  16,  17,  18,  19,  20,  21,
          22,  23,  30,  31,  32,  33,  34,  35,  36,  37,  44,  45,  46,  47,
          48,  49,  50,  51,  58,  59,  60,  61,  62,  63,  64,  65,  72,  73,
          74,  75,  76,  77,  78,  79,  86,  87,  88,  89,  90,  91,  92,  93,
         100, 101, 102, 103, 104, 105, 106, 107, 114, 115, 116, 117, 118, 119,
         12

## Seeing FMOW

In [1]:
import os
from os.path import join
import numpy as np
from tqdm import tqdm
classes = sorted(os.listdir(f'/raid/biplab/datasets/fmow_sentinel/train'))




data = {
    'rgb':{
        'train':[],
        'val':[]
    },
    'sentinel':{
        'train':[],
        'val':[]
    }
}

for split in ['train', 'val']:
    overall_rgb_files = []
    overall_sentinel_files = []
    rgb_path = f'/raid/biplab/datasets/DATA/fmow-rgb/{split}/'
    sentinel_path = f'/raid/biplab/datasets/fmow_sentinel/{split}/'
    common_folders = {}
    for clas in tqdm(classes):
        rgb_clas_path = join(rgb_path, clas)
        sentinel_clas_path = join(sentinel_path, clas)
        common_folders[clas] = []
        for folder in os.listdir(sentinel_clas_path):
            if folder in os.listdir(rgb_clas_path):
                common_folders[clas].append(folder)

    for clas in tqdm(classes):
        rgb_clas_path = join(rgb_path, clas)
        sentinel_clas_path = join(sentinel_path, clas)
        for folder in common_folders[clas]:
            rgb_folder_path = join(rgb_clas_path, folder)
            sentinel_folder_path = join(sentinel_clas_path, folder)
            rgb_files = np.array([file for file in os.listdir(rgb_folder_path) if not file.endswith('.json')])
            sentinel_files = np.array([file for file in os.listdir(sentinel_folder_path) if file.endswith('.tif')])
            for file_sentinel in sentinel_files:
                for file_rgb in rgb_files:
                    if file_sentinel.split('.')[0].replace('_', ' ').strip() == file_rgb.split('.')[0].replace('rgb', '').replace('msrgb', '').replace('_', ' ').strip():
                        overall_rgb_files.append(join(rgb_folder_path, file_rgb))
                        overall_sentinel_files.append(join(sentinel_folder_path, file_sentinel))
    data['rgb'][split] = overall_rgb_files
    data['sentinel'][split] = overall_sentinel_files

print(len(data['rgb']['train']), len(data['sentinel']['train']), len(data['rgb']['val']), len(data['sentinel']['val']))

import json
with open(f'fmow_data/fmow_rgb.json', 'w') as f:
    json.dump(data['rgb'], f)


with open(f'fmow_data/fmow_sentinel.json', 'w') as f:
    json.dump(data['sentinel'], f)

100%|██████████| 62/62 [00:01<00:00, 35.86it/s]


173641 173641 25133 25133


In [2]:
len(data['rgb']['train']), len(data['sentinel']['train']), len(data['rgb']['val']), len(data['sentinel']['val'])

(173641, 173641, 25133, 25133)